In [0]:
dbutils.widgets.text("env", "dev", "env")
env = dbutils.widgets.get("env")
print(env)

In [0]:
import json
import requests
file_path = f"../config/{env}.json"
with open(file_path, "r") as f:
    config = json.load(f)


In [0]:
workspace = config['workspace']
metadata_schema = config['metadata']

In [0]:
source_mapping = {
    'github':config['github'],
    'master':config['master'],
    'finnhub':config['finnhub']
}

In [0]:
target_mapping = {
    'master':config['github'],
    'm101_finnhub':config['m101_finnhub']
}

In [0]:
execution_type = {
    "ingestion": "ni",
    "consumption": "con"
}

In [0]:
def insert_log(param_name, process_name, source_schema, source_table, target_schema, target_table, process_date,process_status, log):
    # error_msg = str(log).replace("'", "''")
    spark.sql(
        f"""
        INSERT INTO {metadata_schema}.audit_log_{env} 
        VALUES ('{param_name}', '{process_name}', '{source_schema}', '{source_table}', '{target_schema}', '{target_table}', '{process_date}', current_timestamp(), '{process_status}', '{log}')
        """
    )

In [0]:
def get_github_events(src):
    try:
        TOKEN = source_mapping[src]
        headers = {
            "Authorization": f"Bearer {TOKEN}",
            "Accept": "application/vnd.github+json"
        }
        url = "https://api.github.com/events"
        response = requests.get(url, headers=headers)
        return response.status_code
    except Exception as e:
        return f"Error: {str(e)}"

In [0]:

def get_finnhub_connection(symbol, api_key):
    url = "https://finnhub.io/api/v1/quote"
    params = {
        "symbol": symbol,
        "token": api_key
    }
    response = requests.get(url, params=params)
    return response.json()

In [0]:
def check_param_name_for_tgt(tgt):
    get_param_details = spark.sql(f"""select * from metadata_{env}.param_table_{env} where param_name='{tgt}'""").toPandas().to_dict('records')
    if len(get_param_details) == 0:
        return False
    else:
        return True



In [0]:
def register_tgt(tgt, execution_type):
    if not check_param_name_for_tgt(tgt):
        spark.sql(f"""insert into metadata_{env}.param_table_{env} values ('{tgt}', 'Y', cast(date_format(current_date(), 'yyyyMMdd') as int),'{execution_type}')""")
        spark.sql(f"""create schema if not exists {workspace}.{target_mapping[tgt]} """)
        return f"target registered successfully"
    else:
        return f"target already registered"
